# Exploración de Riesgos — PISST
**Objetivo:** Explorar peligros, evaluaciones y medidas de control para validar la lógica de `analizar_riesgos()`.

Este notebook es de solo lectura — nunca escribe en la BD.

## 1. Configuración e imports

In [ ]:
import os
import sys

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

sys.path.insert(0, os.path.abspath('..'))
load_dotenv('../.env')

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

print('Pandas:', pd.__version__, '| NumPy:', np.__version__)

## 2. Conexión a la BD

In [ ]:
DATABASE_URL = os.getenv('DATABASE_URL')
engine = create_engine(DATABASE_URL)

with engine.connect() as conn:
    total_peligros = conn.execute(text('SELECT COUNT(*) FROM peligros')).scalar()
    total_evaluaciones = conn.execute(text('SELECT COUNT(*) FROM evaluaciones_riesgo')).scalar()
    total_controles = conn.execute(text('SELECT COUNT(*) FROM medidas_control')).scalar()
    print(f'Peligros: {total_peligros} | Evaluaciones: {total_evaluaciones} | Controles: {total_controles}')

## 3. Carga de datos (Ingesta)

In [ ]:
# Peligros con su última evaluación de riesgo (no residual)
query_peligros = """
SELECT
    p.id             AS peligro_id,
    p.tipo           AS tipo_peligro,
    p.activo,
    p.empresa_id,
    er.nivel_riesgo,
    er.es_residual
FROM peligros p
LEFT JOIN evaluaciones_riesgo er ON er.peligro_id = p.id AND er.es_residual = FALSE
"""

# Medidas de control
query_controles = """
SELECT
    mc.peligro_id,
    mc.estado,
    p.empresa_id
FROM medidas_control mc
JOIN peligros p ON p.id = mc.peligro_id
"""

df_peligros = pd.read_sql(query_peligros, engine)
df_controles = pd.read_sql(query_controles, engine)

print(f'Peligros cargados: {len(df_peligros)}')
print(f'Controles cargados: {len(df_controles)}')
df_peligros.head()

## 4. Inspección estructural

In [ ]:
print('Shape peligros:', df_peligros.shape)
df_peligros.info()
print()
print('Tipos de peligro únicos:', df_peligros['tipo_peligro'].unique())
print('Niveles de riesgo únicos:', df_peligros['nivel_riesgo'].unique())

## 5. Análisis descriptivo

In [ ]:
print('Distribución por nivel de riesgo:')
print(df_peligros['nivel_riesgo'].value_counts())
print()
print('Distribución por tipo de peligro:')
print(df_peligros['tipo_peligro'].value_counts())
print()
print('Estado de medidas de control:')
print(df_controles['estado'].value_counts())

## 6. Transformaciones y cálculos (lógica del servicio)

In [ ]:
if len(df_peligros) == 0:
    print('Sin datos — el servicio retornaría ceros.')
else:
    total_peligros = len(df_peligros)

    # Distribución por nivel de riesgo
    distribucion_nivel = df_peligros['nivel_riesgo'].value_counts().to_dict()

    # % con medidas de control implementadas (estado = 'implementada')
    peligros_con_control = df_controles[df_controles['estado'] == 'implementada']['peligro_id'].nunique()
    pct_con_control = round(peligros_con_control / total_peligros * 100, 1) if total_peligros > 0 else 0.0

    # Peligros críticos sin ninguna medida de control
    peligros_ids_con_control = set(df_controles['peligro_id'].tolist())
    df_criticos = df_peligros[df_peligros['nivel_riesgo'] == 'critico']
    criticos_sin_control = len(df_criticos[~df_criticos['peligro_id'].astype(str).isin(peligros_ids_con_control)])

    # Distribución por tipo de peligro
    distribucion_tipo = df_peligros['tipo_peligro'].value_counts().to_dict()

    resultado = {
        'total_peligros': total_peligros,
        'por_nivel': distribucion_nivel,
        'por_tipo': distribucion_tipo,
        'pct_con_control_implementado': pct_con_control,
        'criticos_sin_control': criticos_sin_control,
    }

    import json
    print(json.dumps(resultado, indent=2, ensure_ascii=False))

## 7. Exportación de hallazgos

In [ ]:
os.makedirs('../data/processed', exist_ok=True)

if len(df_peligros) > 0:
    df_peligros.to_csv('../data/processed/riesgos_explorados.csv', index=False)
    df_controles.to_csv('../data/processed/controles_explorados.csv', index=False)
    print('Exportado a data/processed/')

## 8. Conclusiones para el servicio

- `analizar_riesgos()` necesita dos queries: una para peligros+evaluaciones, otra para controles
- El % de control se calcula sobre `medidas_control.estado == 'implementada'` (no solo existir la medida)
- Los peligros críticos sin ninguna medida son la alerta más importante para SST
- Multi-tenancy: ambas queries filtran por `empresa_id`